<a href="https://colab.research.google.com/github/Datkhoo25/insurance_risk_prediction/blob/main/Deployment_Saving_XGBModel_pkl.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import pickle
from scipy.stats import randint, uniform
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score
from xgboost import XGBClassifier

# Example of how to load the preprocessed data
def load_preprocessed_data(data_path):
    with open(data_path, 'rb') as file:
        preprocessed_data = pickle.load(file)
    return preprocessed_data

# Load the preprocessed data
data_dict = load_preprocessed_data('/content/drive/MyDrive/Colab Notebooks/Risk Prediction/transformed_data.pkl')

X_train_transformed = data_dict['X_train_transformed']
X_val_transformed = data_dict['X_val_transformed']
y_train = data_dict['y_train']
y_val = data_dict['y_val']

# Adjust the target classes to start from 0 instead of 1
y_train_adjusted = y_train - 1
y_val_adjusted = y_val - 1

random_state = 42

# Define the parameter distributions
param_dist = {
    'max_depth': randint(10, 40),  # XGBClassifier parameters
    'min_child_weight': randint(0, 3),
    'subsample': uniform(0.5, 0.4),
    'colsample_bytree': uniform(0.5, 0.4),
    'learning_rate': uniform(0.1, 0.3),
    'n_estimators': randint(500, 1500)
}

xgb = XGBClassifier(objective='multi:softmax', num_class=8, n_jobs=-1)

# Instantiate RandomizedSearchCV
random_search = RandomizedSearchCV(estimator=xgb,
                                   param_distributions=param_dist,
                                   n_iter=30,  # Number of parameter settings that are sampled
                                   scoring='accuracy',
                                   cv=2,
                                   verbose=1,
                                   n_jobs=-1,
                                   random_state=42)

# Fit RandomizedSearchCV to the training data
random_search.fit(X_train_transformed, y_train_adjusted)

# Print the best parameters and best score
print("Best parameters found:", random_search.best_params_)
print("Best cross-validation score:", random_search.best_score_)

# Get the best estimator
best_xgb_model = random_search.best_estimator_

# Predict on validation set with the best estimator
y_pred_adjusted = best_xgb_model.predict(X_val_transformed)

# Adjust the predictions back to the original class range
y_pred = y_pred_adjusted + 1

# Evaluate accuracy with the best estimator
accuracy_best = accuracy_score(y_val, y_pred)
print(f"Accuracy on validation set with best estimator: {accuracy_best:.4f}")

# Evaluate precision with the best estimator
precision_best = precision_score(y_val, y_pred, average='micro')
print(f"Precision on validation set with best estimator: {precision_best:.4f}")

# Evaluate recall with the best estimator
recall_best = recall_score(y_val, y_pred, average='micro')
print(f"Recall on validation set with best estimator: {recall_best:.4f}")

# Save the best XGBoost model using Pickle
with open('/content/drive/MyDrive/Colab Notebooks/Risk Prediction/best_xgb_model.pkl', 'wb') as file:
    pickle.dump(best_xgb_model, file)

Mounted at /content/drive
Fitting 3 folds for each of 50 candidates, totalling 150 fits


/usr/local/lib/python3.11/dist-packages/joblib/externals/loky/process_executor.py:752: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Best parameters found: {'colsample_bytree': 0.7473544037332349, 'learning_rate': 0.21473859738014883, 'max_depth': 29, 'min_child_weight': 0, 'n_estimators': 630, 'subsample': 0.8439761626945282}
Best cross-validation score: 0.5820983815920827
Accuracy on validation set with best estimator: 0.5911
Precision on validation set with best estimator: 0.5911
Recall on validation set with best estimator: 0.5911
